# **Get Legislator Information (Plural)**

This notebook extracts and processes information about **U.S. state retired legislators** from YAML files available in the OpenStates dataset. It converts these YAML files into a single, structured DataFrame, ensures all identifier schemes are included, and saves the result as a CSV file.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd                     # For data manipulation
import json                             # For file handling
import os                               # For file handling
#!pip install pyyaml
import yaml                             # To read .yml files
from collections import defaultdict     # To simplify missing key handling

## **Define Source Folder and List YAML Files**

The dataset consists of multiple .yml files located in a local folder that mirrors the OpenStates “people” repository.
Here, we define the folder path, list all files that end with .yml, and print them to verify that the data has been loaded correctly.

### **Colab Version Importation**

In [4]:
!git clone https://github.com/openstates/people.git

Cloning into 'people'...
remote: Enumerating objects: 342013, done.
remote: Counting objects: 100% (1864/1864), done.
remote: Compressing objects: 100% (1027/1027), done.
remote: Total 342013 (delta 1466), reused 885 (delta 837), pack-reused 340149 (from 3)
Receiving objects: 100% (342013/342013), 97.83 MiB | 22.21 MiB/s, done.
Resolving deltas: 100% (280063/280063), done.
Updating files: 100% (25767/25767), done.


In [5]:
folder_path = "/content/people/data/us/retired"
files = [f for f in os.listdir(folder_path) if f.endswith(".yml")]
print(len(files), "YAML files found")

187 YAML files found


### **Local Version Importation**

In [6]:
# clone repository from github at https://github.com/openstates/people.git
#folder_path = "data/us/legislature"
#files = [f for f in os.listdir(folder_path) if f.endswith(".yml")]
# print the files
#print(files)

## **Initialize Data Structures**
We initialize two key structures:

- `rows`: a list that will hold each legislator’s information as a dictionary.
- `all_schemes`: a set to collect all possible identifier schemes across files (e.g., bioguide, wikidata, votesmart, etc.).

In [7]:
# Convert all the yaml files to a dataframe
rows = []

# First pass: gather all possible identifier schemes
all_schemes = set()

## **First Pass — Gather Identifier Schemes**

During this first loop through all YAML files:

We open each file safely using UTF-8 encoding.

Extract the section "other_identifiers" if it exists.

Collect each unique "scheme" value (these are different ID systems used across legislators).

This ensures that we know **all possible identifier types** before building the DataFrame.

In [8]:
for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
        for id_obj in data.get("other_identifiers", []):
            all_schemes.add(id_obj.get("scheme"))

## **Second Pass — Build Legislator Records**
Now that all identifier schemes are known, we process each YAML file to extract structured information:

- Basic demographic data (name, gender, birth date)

- Role information (district, chamber, start/end dates)

- Party affiliation

- Contact information and social media IDs

- Other identifiers (bioguide, wikidata, etc.)

Each record is stored as a dictionary in `rows`.

In [9]:
# Now process each file with all identifier columns
for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

        # Prepare default row
        row = defaultdict(lambda: None)

        # Basic fields
        row["id"] = data.get("id")
        row["name"] = data.get("name")
        row["given_name"] = data.get("given_name")
        row["family_name"] = data.get("family_name")
        row["birth_date"] = data.get("birth_date")
        row["gender"] = data.get("gender")
        row["email"] = data.get("email")
        row["image"] = data.get("image")
        row["party"] = data.get("party", [{}])[0].get("name")

        # Roles
        role = data.get("roles", [{}])[-1]
        row["role_type"] = role.get("type")
        row["district"] = role.get("district")
        row["role_start_date"] = role.get("start_date")
        row["role_end_date"] = role.get("end_date")

        # Social media (if available)
        for k, v in data.get("ids", {}).items():
            row[f"social_{k}"] = v

        # Other identifiers
        for id_obj in data.get("other_identifiers", []):
            scheme = id_obj.get("scheme")
            identifier = id_obj.get("identifier")
            row[f"{scheme}_id"] = identifier

        rows.append(row)

All individual records are now consolidated into a single pandas DataFrame.
Each row corresponds to one legislator, and each column represents a field or identifier.

In [10]:
# Create DataFrame
df = pd.DataFrame(rows)

## **Add Missing Columns for Unused Schemes**

Some identifier schemes may not appear for all legislators.
This step ensures that **all scheme columns exist**, even if they contain only None values for some legislators.

In [11]:
# Fill in missing columns for any scheme not found in every legislator
for scheme in all_schemes:
    col_name = f"{scheme}_id"
    if col_name not in df.columns:
        df[col_name] = None

In [12]:
df.columns
df.head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,ballotpedia_id,google_entity_id_id,votesmart_id,icpsr_id,thomas_id,cspan_id,house_history_id,maplight_id,pictorial_id,lis_id
0,ocd-person/efc4890a-e0da-5a47-a44d-39145b7d5903,Conor Lamb,Conor,Lamb,1984-06-27,M,None,https://unitedstates.github.io/images/congress...,Democratic,lower,...,Conor Lamb,kg:/g/11gdj36kt1,177656,21756,NaN,NaN,NaN,NaN,NaN,NaN
1,ocd-person/61ef9322-82bc-52a1-983e-3026b8834a43,James R. Langevin,James,Langevin,1964-04-22,M,None,https://unitedstates.github.io/images/congress...,Democratic,lower,...,Jim Langevin,kg:/m/02tvqy,55787,20136,01668,86608,17296,331,NaN,NaN
2,ocd-person/716b011f-3673-558d-b35f-d5a454cafa81,Gerry Connolly,Gerry,Connolly,1950-03-30,Male,https://connolly.house.gov/contact/zip-auth.htm,https://unitedstates.github.io/images/congress...,Democratic,lower,...,Gerald Connolly,kg:/m/04f40kn,95078,20952,01959,1015936,11817,754,13309,NaN
3,ocd-person/5a8792bc-1520-56aa-ab19-729008cd559b,JD Vance,JD,Vance,1984-08-02,Male,https://www.vance.senate.gov/contact/,https://unitedstates.github.io/images/congress...,Republican,upper,...,J.D. Vance,kg:/g/11c6v_wj1r,201794,NaN,NaN,NaN,NaN,NaN,13348,S421
4,ocd-person/b5356530-dd09-5b14-98f2-365d70bf4e5c,Susan Wild,Susan,Wild,1957-06-07,Female,https://wild.house.gov/address_authentication?...,https://unitedstates.github.io/images/congress...,Democratic,lower,...,Susan Wild,kg:/g/11gtg86jn6,178895,21763,NaN,NaN,NaN,NaN,13227,NaN


To verify data completeness, we check if any legislators lack a bioguide_id.
The resulting DataFrame displays those records for review.

In [13]:
# are there any missing bioguide ids?
df[df['bioguide_id'].isna()].head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,ballotpedia_id,google_entity_id_id,votesmart_id,icpsr_id,thomas_id,cspan_id,house_history_id,maplight_id,pictorial_id,lis_id
126,ocd-person/21f03982-e598-4407-a0c6-3451f62afcf9,Marlin Stutzman,Marlin,Stutzman,1976-08-31,Male,marlin@stutzmangroup.us,https://storage.googleapis.com/enview-dev-publ...,Republican,lower,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# bioguide_id missing - Marlin Stutzman
df.loc[df['name'] == 'Marlin Stutzman', 'bioguide_id'] = 'S001188'

In [15]:
df[df['bioguide_id'].isna()]

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,ballotpedia_id,google_entity_id_id,votesmart_id,icpsr_id,thomas_id,cspan_id,house_history_id,maplight_id,pictorial_id,lis_id


## **Export to CSV**

Finally, we export the full DataFrame to a CSV file named `plural_retired_legislators_with_bioguide.csv`, keeping all columns intact and disabling the index column.

In [16]:
# Define the project root directory (Google Drive)
drive_root = "/content/drive/Shareddrives/Data Archive/Bridge_Grades"

# Define the relative output path from the project root
base_dir = os.path.join(
    drive_root,
    "Input_Files",
    "Snapshot_Raw_2026_06_02",
    "Raw_Others_2026_06_02"
)

# Check if the directory exists (do not create it)
if not os.path.isdir(base_dir):
    raise OSError(f"Directory does not exist: {base_dir}")

# Define the output file name and full path
output_filename = "plural_retired_legislators_with_bioguide.csv"
output_path = os.path.join(base_dir, output_filename)

# Save the DataFrame to CSV
df.to_csv(output_path, index=False)

# Display confirmation message
print(f"File saved as: {output_path}")

File saved as: /content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/plural_retired_legislators_with_bioguide.csv
